In [15]:
import os
import sys
import numpy as np
import pandas as pd
import librosa
import librosa.effects
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization, Add
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# --- Step 1: Configuration & Parameters ---
LABEL_PATH_2_3 = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
# Make sure this AUDIO_DIR_2_3 points to the folder containing Stage 2 & 3 patient audio
AUDIO_DIR_2_3 = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
MODEL_SAVE_PATH_2_3 = "best_copd_2_3_advanced_model.keras"

# --- Training Parameters with new Learning Rate ---
N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 128, 150, 15, 32
INITIAL_LEARNING_RATE = 0.0001 # Reduced for more careful training

# --- Augmentation Parameters ---
NOISE_FACTOR, TIME_SHIFT_MAX_SEC, PITCH_SHIFT_STEPS, TIME_STRETCH_RATE = 0.005, 0.2, 4, 0.8


# --- Step 2: Helper Functions (with SpecAugment added) ---
def add_gaussian_noise(y, noise_factor=NOISE_FACTOR): return y + noise_factor * np.random.randn(len(y))
def time_shift(y, sr, shift_max_sec=TIME_SHIFT_MAX_SEC): return np.roll(y, int(sr*np.random.uniform(-shift_max_sec, shift_max_sec)))
def extract_log_mel_spectrogram(y, sr):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel_spec)
    if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0, 0), (0, MAX_LEN - log_mel.shape[1])), mode='constant')
    else: log_mel = log_mel[:, :MAX_LEN]
    return log_mel

def spec_augment(spectrogram, time_masking_para=30, frequency_masking_para=20, num_time_masks=1, num_freq_masks=1):
    """
    A manual implementation of SpecAugment.
    Masks random frequency and time bands in the spectrogram.
    """
    spectrogram_copy = np.copy(spectrogram)
    for _ in range(num_freq_masks):
        f = np.random.uniform(low=0.0, high=frequency_masking_para)
        f = int(f)
        f0 = np.random.randint(0, N_MELS - f)
        spectrogram_copy[f0:f0 + f, :] = 0
    for _ in range(num_time_masks):
        t = np.random.uniform(low=0.0, high=time_masking_para)
        t = int(t)
        t0 = np.random.randint(0, MAX_LEN - t)
        spectrogram_copy[:, t0:t0 + t] = 0
    return spectrogram_copy

# --- Step 3: Load Data and Apply Oversampling ---
print("--- Step 3: Loading Data for COPD 2 vs 3 ---")
df = pd.read_excel(LABEL_PATH_2_3)
df_copd_2_3 = df[df["Diagnosis"].isin(["COPD2", "COPD3"])].copy()

if df_copd_2_3['Diagnosis'].nunique() < 2:
    raise ValueError("Excel file must contain patients from 'COPD2' and 'COPD3'.")

df_copd_2_3['label_encoded'] = df_copd_2_3['Diagnosis'].apply(lambda x: 0 if x == 'COPD2' else 1)
label_dict_2_3 = dict(zip(df_copd_2_3["Patient ID"], df_copd_2_3["label_encoded"]))
patient_ids_2_3 = list(label_dict_2_3.keys())
patient_labels_2_3 = list(label_dict_2_3.values())

print("\nPerforming stratified patient-aware split...")
try:
    train_pids, test_pids, y_train_pids_labels, _ = train_test_split(
        patient_ids_2_3, patient_labels_2_3,
        test_size=0.25, random_state=42, stratify=patient_labels_2_3
    )
except ValueError as e:
    raise ValueError(f"\nFATAL ERROR: {e}\nThis means a class has only 1 patient.") from e

# Oversampling Logic
print("\nCalculating oversampling rate...")
base_augmentations = 5
num_copd2_train = y_train_pids_labels.count(0)
num_copd3_train = y_train_pids_labels.count(1)
if num_copd2_train < num_copd3_train: minority_label, minority_count, majority_count = 0, num_copd2_train, num_copd3_train
else: minority_label, minority_count, majority_count = 1, num_copd3_train, num_copd2_train
augmentations_for_minority = round(base_augmentations * (majority_count / minority_count)) if minority_count > 0 else base_augmentations
print(f"Patient distribution: {num_copd2_train} COPD2, {num_copd3_train} COPD3. Augmentations: Majority={base_augmentations}, Minority={augmentations_for_minority}.")

X_train, y_train, X_test, y_test = [], [], [], []
for pid in patient_ids_2_3:
    label = label_dict_2_3[pid]
    for side in ['L', 'R']:
        for i in range(1, 7):
            fpath = os.path.join(AUDIO_DIR_2_3, f"{pid}_{side}{i}.wav")
            if not os.path.exists(fpath): continue
            try:
                y_audio, sr = librosa.load(fpath, sr=None)
                if pid in train_pids:
                    num_augs = augmentations_for_minority if label == minority_label else base_augmentations
                    y_train.extend([label] * num_augs)
                    original_spec = extract_log_mel_spectrogram(y_audio, sr)
                    augs = [original_spec, spec_augment(original_spec), extract_log_mel_spectrogram(add_gaussian_noise(y_audio), sr), extract_log_mel_spectrogram(time_shift(y_audio, sr), sr), extract_log_mel_spectrogram(librosa.effects.pitch_shift(y=y_audio, sr=sr, n_steps=PITCH_SHIFT_STEPS), sr), extract_log_mel_spectrogram(librosa.effects.time_stretch(y=y_audio, rate=1/TIME_STRETCH_RATE), sr), spec_augment(original_spec)]
                    X_train.extend(augs[:num_augs])
                elif pid in test_pids:
                    X_test.append(extract_log_mel_spectrogram(y_audio, sr))
                    y_test.append(label)
            except Exception as e:
                print(f"Warning: Error processing {fpath}: {e}")

X_train, y_train = np.array(X_train), np.array(y_train)
X_test, y_test = np.array(X_test), np.array(y_test)
if len(np.unique(y_train)) < 2: raise ValueError(f"FATAL: Training set has <2 classes due to MISSING audio files in '{AUDIO_DIR_2_3}'.")
X_train, X_test = X_train[..., np.newaxis], X_test[..., np.newaxis]
print("\n--- Final Data Shapes ---\n", f"X_train: {X_train.shape}, y_train: {y_train.shape}\n", f"X_test: {X_test.shape}, y_test: {y_test.shape}")


# --- Step 4: Verify Class Balance ---
print("\n--- Step 4: Verifying Data Balance ---")
print(f"Final training sample counts: COPD2(0)={np.sum(y_train == 0)}, COPD3(1)={np.sum(y_train == 1)}")


# --- Step 5: Build the Regularized ResNet-like Model ---
print("\n--- Step 5: Building Regularized ResNet-like Model for 2-3 Progression ---")
def resnet_block(input_tensor, filters):
    x = Conv2D(filters, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001))(input_tensor)
    x = BatchNormalization()(x)
    x = Conv2D(filters, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Add()([x, input_tensor])
    return x

input_layer = Input(shape=(N_MELS, MAX_LEN, 1))
x = Conv2D(64, (3, 3), activation='relu', padding='same')(input_layer)
x = BatchNormalization()(x)
x = resnet_block(x, filters=64); x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x); x = Dropout(0.4)(x)
x = resnet_block(x, filters=64); x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x); x = Dropout(0.4)(x)
x = Flatten()(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x); x = Dropout(0.5)(x)
output_layer = Dense(1, activation='sigmoid')(x)
model_2_3 = Model(inputs=input_layer, outputs=output_layer)
model_2_3.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model_2_3.summary()


--- Step 3: Loading Data for COPD 2 vs 3 ---

Performing stratified patient-aware split...

Calculating oversampling rate...
Patient distribution: 5 COPD2, 5 COPD3. Augmentations: Majority=5, Minority=5.

--- Final Data Shapes ---
 X_train: (600, 128, 150, 1), y_train: (600,)
 X_test: (48, 128, 150, 1), y_test: (48,)

--- Step 4: Verifying Data Balance ---
Final training sample counts: COPD2(0)=300, COPD3(1)=300

--- Step 5: Building Regularized ResNet-like Model for 2-3 Progression ---


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 128, 150,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_37 (Conv2D)  │ (None, 128, 150,  │        640 │ input_layer_5[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_37[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_38 (Conv2D)  │ (None, 128, 150,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_38[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_39 (Conv2D)  │ (None, 128, 150,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_39[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 128, 150,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_40 (Conv2D)  │ (None, 128, 150,  │     36,928 │ add_16[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_40[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_41 (Conv2D)  │ (None, 128, 150,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_41[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 128, 150,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ add_16[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 64, 75,    │          0 │ add_17[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 64, 75,    │          0 │ max_pooling2d_8[… │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_42 (Conv2D)  │ (None, 64, 75,    │     36,928 │ dropout_12[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 75,    │        256 │ conv2d_42[0][0] 

 Total params: 9,997,953 (38.14 MB)

 Trainable params: 9,996,801 (38.13 MB)

 Non-trainable params: 1,152 (4.50 KB)

In [16]:
# --- Step 6: Train the Model ---
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
model_checkpoint = ModelCheckpoint(MODEL_SAVE_PATH_2_3, save_best_only=True, monitor='val_accuracy', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
print(f"\n--- Step 6: Starting Model Training for COPD 2-3 on Balanced Data ---\n")
history_2_3 = model_2_3.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, model_checkpoint, reduce_lr])


--- Step 6: Starting Model Training for COPD 2-3 on Balanced Data ---

Epoch 1/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.5355 - loss: 6.6484
Epoch 1: val_accuracy improved from -inf to 0.56250, saving model to best_copd_2_3_advanced_model.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 114s 6s/step - accuracy: 0.5365 - loss: 6.5431 - val_accuracy: 0.5625 - val_loss: 1.4226 - learning_rate: 1.0000e-04
Epoch 2/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.6396 - loss: 1.4406
Epoch 2: val_accuracy did not improve from 0.56250
19/19 ━━━━━━━━━━━━━━━━━━━━ 113s 6s/step - accuracy: 0.6399 - loss: 1.4407 - val_accuracy: 0.5000 - val_loss: 2.3606 - learning_rate: 1.0000e-04
Epoch 3/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7132 - loss: 1.3315
Epoch 3: val_accuracy did not improve from 0.56250
19/19 ━━━━━━━━━━━━━━━━━━━━ 116s 6s/step - accuracy: 0.7125 - loss: 1.3322 - val_accuracy: 0.5000 - val_loss: 1.4609 - learning_rate: 1.0000e-04
Epoch 4/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [17]:
# --- Step 7: Evaluate the Final 2-3 Model ---
print("\n--- Step 7: Evaluating Best Saved 2-3 Model ---")
# The best model's weights from the training run are already loaded due to
# 'restore_best_weights=True' in EarlyStopping. We can also explicitly load
# the checkpointed model to be certain.
model_2_3.load_weights(MODEL_SAVE_PATH_2_3)

# Evaluate on the unseen test data
loss, accuracy = model_2_3.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy (2 vs 3): {accuracy*100:.2f}%")
print(f"Final Test Loss (2 vs 3): {loss:.4f}")


--- Step 7: Evaluating Best Saved 2-3 Model ---

Final Test Accuracy (2 vs 3): 62.50%
Final Test Loss (2 vs 3): 1.4016


In [18]:
# --- Step 8: Patient-Level Analysis (Averaging Method for 2-3) ---

def run_patient_analysis_by_average(trained_model, labels_path, audio_dir, target_diagnosis, sort_ascending):
    """
    Analyzes patients by averaging scores and shows the raw prediction for each audio file.
    This function is adapted for the 2-vs-3 analysis.
    """
    print(f"\n--- 📈 Starting Analysis for '{target_diagnosis}' Patients (Averaging Scores) ---")
    df_labels = pd.read_excel(labels_path)
    # Filter for the specific diagnosis being analyzed
    patient_ids = df_labels[df_labels['Diagnosis'] == target_diagnosis]['Patient ID'].tolist()
    if not patient_ids: return None
    
    print(f"Found {len(patient_ids)} patients with diagnosis '{target_diagnosis}'. Analyzing files...")
    results_list = []
    
    for pid in patient_ids:
        scores = []
        for side in ['L', 'R']:
            for i in range(1, 7):
                fpath = os.path.join(audio_dir, f"{pid}_{side}{i}.wav")
                if not os.path.exists(fpath): continue
                try:
                    y, sr = librosa.load(fpath, sr=None)
                    spec = extract_log_mel_spectrogram(y, sr)
                    prob = trained_model.predict(np.expand_dims(spec, axis=(0, -1)), verbose=0)[0][0]
                    scores.append(prob)
                except Exception as e:
                    print(f"Warning: Could not process file {fpath}: {e}")
        
        if scores:
            # Print the detailed scores for this patient
            formatted_scores = [f'{s:.4f}' for s in scores]
            print(f"  -> Raw Scores for Patient {pid}: {formatted_scores}")
            
            # Append the calculated average to the results list
            results_list.append({'Patient ID': pid, 'Avg_Score': np.mean(scores), 'Audio_Files_Found': len(scores)})
            
    if not results_list:
        print(f"Warning: Could not generate any analysis results for {target_diagnosis}.")
        return None

    results_df = pd.DataFrame(results_list).sort_values(by='Avg_Score', ascending=sort_ascending)
    return results_df.reset_index(drop=True)

# Define new assessment functions specific to the 2-vs-3 task
def assess_copd3_patient(row):
    """Generates assessment text for a patient known to have COPD3."""
    # A score >= 0.5 means the model correctly leans towards a COPD3 diagnosis.
    return "✅ Model Confident (Correctly resembles Stage 3)" if row['Avg_Score'] >= 0.5 else "⚠️ Model Lacks Confidence (False Negative)"

def assess_copd2_progression_risk(row):
    """Generates assessment text for a Stage 2 patient, checking for risk of progressing to Stage 3."""
    # A score >= 0.5 for a COPD2 patient flags them as high risk.
    return "⚠️ High Risk (Resembles Stage 3 - False Positive)" if row['Avg_Score'] >= 0.5 else "✅ Low Risk (Correctly identified as Stage 2)"


# --- Analyze COPD3 Patients (The "progressed" group for this model) ---
copd3_avg_df = run_patient_analysis_by_average(model_2_3, LABEL_PATH_2_3, AUDIO_DIR_2_3, 'COPD3', True)
if copd3_avg_df is not None:
    copd3_avg_df['Assessment'] = copd3_avg_df.apply(assess_copd3_patient, axis=1)
    print("\n--- ✅ COPD3 Patient Confidence Results (Averaging Method) ---")
    print("Shows model's confidence in identifying patients known to have Stage 3. Avg_Score >= 0.5 is correct.")
    print(copd3_avg_df.to_string())

# --- Analyze COPD2 Patients (The "baseline" or risk group for this model) ---
copd2_avg_df = run_patient_analysis_by_average(model_2_3, LABEL_PATH_2_3, AUDIO_DIR_2_3, 'COPD2', False)
if copd2_avg_df is not None:
    copd2_avg_df['Assessment'] = copd2_avg_df.apply(assess_copd2_progression_risk, axis=1)
    print("\n--- ⚠️ COPD2 Patient Progression Risk Results (Averaging Method) ---")
    print("Shows which Stage 2 patients are flagged as being at high risk of progressing to Stage 3.")
    print(copd2_avg_df.to_string())


--- 📈 Starting Analysis for 'COPD3' Patients (Averaging Scores) ---
Found 7 patients with diagnosis 'COPD3'. Analyzing files...
  -> Raw Scores for Patient H007: ['0.5504', '0.8783', '0.4416', '0.5039', '0.6339', '0.8176', '0.5013', '0.9102', '0.3248', '0.4868', '0.5081', '0.7907']
  -> Raw Scores for Patient H008: ['0.5818', '0.7137', '0.7120', '0.7681', '0.9249', '0.8820', '0.7543', '0.9181', '0.7763', '0.6931', '0.9339', '0.5472']
  -> Raw Scores for Patient H010: ['0.6815', '0.7337', '0.6146', '0.6113', '0.3462', '0.7502', '0.5764', '0.6615', '0.5882', '0.6475', '0.3157', '0.4963']
  -> Raw Scores for Patient H026: ['0.9067', '0.8707', '0.9302', '0.7446', '0.9133', '0.8675', '0.9372', '0.9338', '0.9259', '0.7582', '0.9210', '0.8210']
  -> Raw Scores for Patient H033: ['0.8531', '0.9407', '0.5151', '0.6171', '0.6035', '0.6205', '0.8536', '0.9595', '0.5699', '0.6411', '0.4844', '0.4232']
  -> Raw Scores for Patient H034: ['0.8404', '0.4667', '0.8489', '0.5616', '0.8446', '0.8126', '

In [19]:
def run_patient_analysis_by_vote(trained_model, labels_path, audio_dir, target_diagnosis, sort_ascending):
    print(f"\n--- 🗳️ Starting Analysis for '{target_diagnosis}' Patients (Majority Vote) ---")
    df_labels = pd.read_excel(labels_path)
    patient_ids = df_labels[df_labels['Diagnosis'] == target_diagnosis]['Patient ID'].tolist()
    if not patient_ids: return None
    print(f"Found {len(patient_ids)} patients. Analyzing files and counting votes...")
    patient_scores_data = {}
    for pid in patient_ids:
        scores = []
        for side in ['L', 'R']:
            for i in range(1, 7):
                fpath = os.path.join(audio_dir, f"{pid}_{side}{i}.wav")
                if not os.path.exists(fpath): continue
                try:
                    y, sr = librosa.load(fpath, sr=None)
                    spec = extract_log_mel_spectrogram(y, sr)
                    prob = trained_model.predict(np.expand_dims(spec, axis=(0, -1)), verbose=0)[0][0]
                    scores.append(prob)
                except Exception as e:
                    print(f"Warning: Could not process file {fpath}: {e}")
        if scores:
            formatted_scores = [f'{s:.4f}' for s in scores]
            print(f"  -> Raw Scores for Patient {pid}: {formatted_scores}")
            patient_scores_data[pid] = scores
    if not patient_scores_data: return None
    results_list = []
    for pid, scores in patient_scores_data.items():
        copd3_votes = sum(1 for s in scores if s >= 0.5)
        copd2_votes = len(scores) - copd3_votes
        final_prediction = 'COPD3' if copd3_votes > copd2_votes else 'COPD2'
        results_list.append({'Patient ID': pid, 'COPD3_Votes': copd3_votes, 'COPD2_Votes': copd2_votes, 'Total_Files': len(scores), 'Final_Prediction': final_prediction})
    results_df = pd.DataFrame(results_list).sort_values(by='COPD3_Votes', ascending=sort_ascending)
    return results_df.reset_index(drop=True)

def assess_prediction_vs_truth(row, true_label):
    return f"✅ Correct (Predicted {row['Final_Prediction']})" if row['Final_Prediction'] == true_label else f"❌ Incorrect (Predicted {row['Final_Prediction']}, but was {true_label})"

# --- Analyze COPD3 Patients ---
copd3_vote_df = run_patient_analysis_by_vote(model_2_3, LABEL_PATH_2_3, AUDIO_DIR_2_3, 'COPD3', True)
if copd3_vote_df is not None:
    copd3_vote_df['Assessment'] = copd3_vote_df.apply(assess_prediction_vs_truth, true_label='COPD3', axis=1)
    print("\n--- ✅ COPD3 Patient Majority Vote Results ---")
    print(copd3_vote_df.to_string())

# --- Analyze COPD2 Patients ---
copd2_vote_df = run_patient_analysis_by_vote(model_2_3, LABEL_PATH_2_3, AUDIO_DIR_2_3, 'COPD2', False)
if copd2_vote_df is not None:
    copd2_vote_df['Assessment'] = copd2_vote_df.apply(assess_prediction_vs_truth, true_label='COPD2', axis=1)
    print("\n--- ⚠️ COPD2 Patient Progression Risk Results ---")
    print("Incorrect '❌' assessments here are Stage 2 patients the model thinks have progressed.")
    print(copd2_vote_df.to_string())


--- 🗳️ Starting Analysis for 'COPD3' Patients (Majority Vote) ---
Found 7 patients. Analyzing files and counting votes...
  -> Raw Scores for Patient H007: ['0.5504', '0.8783', '0.4416', '0.5039', '0.6339', '0.8176', '0.5013', '0.9102', '0.3248', '0.4868', '0.5081', '0.7907']
  -> Raw Scores for Patient H008: ['0.5818', '0.7137', '0.7120', '0.7681', '0.9249', '0.8820', '0.7543', '0.9181', '0.7763', '0.6931', '0.9339', '0.5472']
  -> Raw Scores for Patient H010: ['0.6815', '0.7337', '0.6146', '0.6113', '0.3462', '0.7502', '0.5764', '0.6615', '0.5882', '0.6475', '0.3157', '0.4963']
  -> Raw Scores for Patient H026: ['0.9067', '0.8707', '0.9302', '0.7446', '0.9133', '0.8675', '0.9372', '0.9338', '0.9259', '0.7582', '0.9210', '0.8210']
  -> Raw Scores for Patient H033: ['0.8531', '0.9407', '0.5151', '0.6171', '0.6035', '0.6205', '0.8536', '0.9595', '0.5699', '0.6411', '0.4844', '0.4232']
  -> Raw Scores for Patient H034: ['0.8404', '0.4667', '0.8489', '0.5616', '0.8446', '0.8126', '0.8397

In [1]:
# =================================================================================
#
#       COPD Progression Model: Stage 2 vs. Stage 3 (N_MELS = 20)
#
# This script trains the ResNet model using a more compact input feature by
# setting the number of Mel bands (n_mels) to 20.
#
# =================================================================================

import os
import sys
import numpy as np
import pandas as pd
import librosa
import librosa.effects
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization, Add
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import random

# --- Step 1: Configuration & Parameters ---
LABEL_PATH_2_3 = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR_2_3 = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
MODEL_SAVE_PATH_2_3 = "testing_2_3_mel20.keras" # New name for this experiment

# --- UPDATED PARAMETER ---
N_MELS = 20 # Changed from 128 to 20
# -----------------------

MAX_LEN, EPOCHS, BATCH_SIZE = 150, 25, 32
INITIAL_LEARNING_RATE = 0.0001
NOISE_FACTOR, TIME_SHIFT_MAX_SEC, PITCH_SHIFT_STEPS, TIME_STRETCH_RATE = 0.005, 0.2, 4, 0.8


# --- Step 2: Helper Functions ---
# Note: extract_log_mel_spectrogram will now use N_MELS = 20 automatically
def add_gaussian_noise(y, sr, noise_factor=NOISE_FACTOR): return y + noise_factor * np.random.randn(len(y))
def time_shift(y, sr, shift_max_sec=TIME_SHIFT_MAX_SEC): return np.roll(y, int(sr*np.random.uniform(-shift_max_sec, shift_max_sec)))
def pitch_shift(y, sr, n_steps=PITCH_SHIFT_STEPS): return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
def time_stretch(y, sr, rate=TIME_STRETCH_RATE): return librosa.effects.time_stretch(y, rate=rate)
def extract_log_mel_spectrogram(y, sr):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS) # Uses the new global N_MELS
    log_mel = librosa.power_to_db(mel_spec)
    if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0, 0), (0, MAX_LEN - log_mel.shape[1])), mode='constant')
    else: log_mel = log_mel[:, :MAX_LEN]
    return log_mel
AUGMENTATION_FUNCTIONS = [add_gaussian_noise, time_shift, pitch_shift, time_stretch]


# --- Step 3: Load Data, Oversample, and Normalize ---
print("--- Step 3: Loading Data for COPD 2 vs 3 ---")
df = pd.read_excel(LABEL_PATH_2_3)
df_copd_2_3 = df[df["Diagnosis"].isin(["COPD2", "COPD3"])].copy()
if df_copd_2_3['Diagnosis'].nunique() < 2: raise ValueError("Excel file must contain 'COPD2' and 'COPD3'.")

df_copd_2_3['label_encoded'] = df_copd_2_3['Diagnosis'].apply(lambda x: 0 if x == 'COPD2' else 1)
label_dict_2_3 = dict(zip(df_copd_2_3["Patient ID"], df_copd_2_3["label_encoded"]))
patient_ids_2_3, patient_labels_2_3 = list(label_dict_2_3.keys()), list(label_dict_2_3.values())

print("\nPerforming stratified patient-aware split...")
try:
    train_pids, test_pids, y_train_pids_labels, _ = train_test_split(patient_ids_2_3, patient_labels_2_3, test_size=0.25, random_state=42, stratify=patient_labels_2_3)
except ValueError as e: raise ValueError(f"\nFATAL ERROR: {e}\nThis means a class has only 1 patient.") from e

train_files_map = []
for pid in train_pids:
    label = label_dict_2_3[pid]
    paths = [os.path.join(AUDIO_DIR_2_3, f) for f in os.listdir(AUDIO_DIR_2_3) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        if os.path.exists(path):
            train_files_map.append({'path': path, 'label': label})
train_df = pd.DataFrame(train_files_map)

print("\nCalculating oversampling ratios...")
train_class_counts = train_df['label'].value_counts()
majority_class_count = train_class_counts.max()
augmentation_ratios = (majority_class_count / train_class_counts).round().astype(int) - 1
for i, ratio in augmentation_ratios.items(): print(f"  Class {i}: {ratio} extra augmentations needed.")

print("\nGenerating final datasets...")
X_train, y_train, X_test, y_test = [], [], [], []
for _, row in train_df.iterrows():
    try:
        y_audio, sr = librosa.load(row['path'], sr=None)
        label, num_augs = row['label'], augmentation_ratios.get(label, 0)
        X_train.append(extract_log_mel_spectrogram(y_audio, sr)); y_train.append(label)
        for _ in range(num_augs):
            aug_func = random.choice(AUGMENTATION_FUNCTIONS)
            y_augmented = aug_func(y_audio.copy(), sr); X_train.append(extract_log_mel_spectrogram(y_augmented, sr)); y_train.append(label)
    except Exception as e: print(f"Warning: Could not process {row['path']}: {e}")
for pid in test_pids:
    label = label_dict_2_3[pid]
    paths = [os.path.join(AUDIO_DIR_2_3, f) for f in os.listdir(AUDIO_DIR_2_3) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        if os.path.exists(path):
            try:
                y_audio, sr = librosa.load(path, sr=None); X_test.append(extract_log_mel_spectrogram(y_audio, sr)); y_test.append(label)
            except Exception as e: print(f"Warning: Could not process test file {path}: {e}")

X_train_raw = np.array(X_train); train_mean = np.mean(X_train_raw, axis=0); train_std = np.std(X_train_raw, axis=0)
print("\nSaving normalization statistics for this model...")
np.save("logmel_testing_2_3_mel20_mean.npy", train_mean)
np.save("logmel_testing_2_3_mel20_std.npy", train_std)
X_train = (X_train_raw - train_mean) / (train_std + 1e-6)
X_test = (np.array(X_test) - train_mean) / (train_std + 1e-6)
X_train, X_test = X_train[..., np.newaxis], X_test[..., np.newaxis]
y_train, y_test = np.array(y_train), np.array(y_test)
print("✅ Stats saved. Data prepared.")


# --- Step 4: Build the ResNet-like Model ---
print("\n--- Step 4: Building the ResNet-like Model for 2-3 Progression ---")
def resnet_block(input_tensor, filters):
    x = Conv2D(filters, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001))(input_tensor); x = BatchNormalization()(x)
    x = Conv2D(filters, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001))(x); x = BatchNormalization()(x)
    return Add()([x, input_tensor])

input_layer = Input(shape=(N_MELS, MAX_LEN, 1)) # This automatically uses N_MELS = 20
x = Conv2D(64, (3, 3), activation='relu', padding='same')(input_layer); x = BatchNormalization()(x)
x = resnet_block(x, filters=64); x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x); x = Dropout(0.4)(x)
x = resnet_block(x, filters=64); x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x); x = Dropout(0.4)(x)
x = Flatten()(x); x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x); x = Dropout(0.5)(x)
output_layer = Dense(1, activation='sigmoid')(x)
model_2_3 = Model(inputs=input_layer, outputs=output_layer)
model_2_3.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model_2_3.summary()


# --- Step 5: Train the Model ---
print(f"\n--- Step 5: Starting Model Training ---\n")
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
model_checkpoint = ModelCheckpoint(MODEL_SAVE_PATH_2_3, save_best_only=True, monitor='val_accuracy', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
history_2_3 = model_2_3.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, model_checkpoint, reduce_lr])


# --- Step 6: Evaluate the Final Model ---
print("\n--- Step 6: Evaluating Best Saved Model ---")
model_2_3.load_weights(MODEL_SAVE_PATH_2_3)
loss, accuracy = model_2_3.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy (2 vs 3, N_MELS=20): {accuracy*100:.2f}%")
print(f"Final Test Loss: {loss:.4f}")
print(f"\n✅ Training complete. Model saved to '{MODEL_SAVE_PATH_2_3}'")

2025-07-13 19:31:11.868424: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 19:31:11.877308: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 19:31:11.898660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752415271.931709    8505 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752415271.941940    8505 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752415271.967135    8505 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

--- Step 3: Loading Data for COPD 2 vs 3 ---

Performing stratified patient-aware split...

Calculating oversampling ratios...
  Class 0: 0 extra augmentations needed.
  Class 1: 0 extra augmentations needed.

Generating final datasets...

Saving normalization statistics for this model...
✅ Stats saved. Data prepared.

--- Step 4: Building the ResNet-like Model for 2-3 Progression ---


2025-07-13 19:31:19.367486: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20, 150,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 20, 150,   │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 20, 150,   │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 20, 150,   │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 20, 150,   │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 150,   │          0 │ batch_normalizat… │
│                     │ 64)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 20, 150,   │     36,928 │ add[0][0]         │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 20, 150,   │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 20, 150,   │          0 │ batch_normalizat… │
│                     │ 64)               │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 10, 75,    │          0 │ add_1[0][0]       │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 75,    │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 10, 75,    │     36,928 │ dropout[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 75,    │        256 │ conv2d_5[0][0]  

 Total params: 1,814,145 (6.92 MB)

 Trainable params: 1,812,993 (6.92 MB)

 Non-trainable params: 1,152 (4.50 KB)


--- Step 5: Starting Model Training ---

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 976ms/step - accuracy: 0.4625 - loss: 7.2478
Epoch 1: val_accuracy improved from -inf to 0.54167, saving model to testing_2_3_mel20.keras
4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.4733 - loss: 7.0134 - val_accuracy: 0.5417 - val_loss: 1.4526 - learning_rate: 1.0000e-04
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 917ms/step - accuracy: 0.5495 - loss: 4.8864
Epoch 2: val_accuracy improved from 0.54167 to 0.58333, saving model to testing_2_3_mel20.keras
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5479 - loss: 4.9307 - val_accuracy: 0.5833 - val_loss: 1.4392 - learning_rate: 1.0000e-04
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 907ms/step - accuracy: 0.6513 - loss: 2.6183
Epoch 3: val_accuracy did not improve from 0.58333
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.6494 - loss: 2.6763 - val_accuracy: 0.4792 - val_loss: 1.4369 - learning_rate: 1.0000e-04
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 913ms/

Training 2.0


In [ ]:
# # =========================================================================
# #
# #       Universal Expert Model Trainer (Definitive, Corrected Version)
# #
# # This script contains the corrected GroupNormalization layer to fix the
# # 'TypeError: unexpected keyword argument 'n'' bug.
# #
# # =========================================================================

# import os
# import sys
# import numpy as np
# import pandas as pd
# import librosa
# import librosa.effects
# import tensorflow as tf
# from sklearn.model_selection import train_test_split
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Layer
# from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
# from tensorflow.keras.regularizers import l2
# import random

# # ===============================================================
# #                !!! TASK CONFIGURATION !!!
# # ===============================================================
# # Run this script 4 times, changing these two variables each time.
# # Example for the first run:
# CLASSES_TO_TRAIN = ['COPD2', 'COPD3']
# MODEL_TO_SAVE_AS = "testing_2_3.keras"
# # ===============================================================

# # --- General Configuration ---
# LABEL_PATH = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
# AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
# SAVE_DIR = "/home/punith/Desktop/cHEAL 2.o/All models"

# N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 20, 150, 30, 32
# INITIAL_LEARNING_RATE = 0.0001


# # --- Step 2: Helper Functions & CORRECTED Custom Layer ---
# def augment(y, sr):
#     """Applies a random augmentation to the audio signal."""
#     if random.random() < 0.5:
#         rate = random.uniform(0.9, 1.1)
#         return librosa.effects.time_stretch(y, rate=rate)
#     else:
#         steps = random.randint(-2, 2)
#         return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

# def extract_log_mel_spectrogram(path, do_augment=False):
#     """Loads an audio file and converts it to a Log-Mel Spectrogram."""
#     y, sr = librosa.load(path, sr=None)
#     if do_augment:
#         y = augment(y, sr)
    
#     mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
#     log_mel = librosa.power_to_db(mel_spec)
    
#     if log_mel.shape[1] < MAX_LEN:
#         log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
#     else:
#         log_mel = log_mel[:, :MAX_LEN]
#     return log_mel

# # --- CORRECTED GroupNormalization Layer ---
# class GroupNormalization(Layer):
#     """Custom Group Normalization layer with correct arguments."""
#     def __init__(self, groups=4, epsilon=1e-5, **kwargs):
#         super(GroupNormalization, self).__init__(**kwargs)
#         self.groups = groups
#         self.epsilon = epsilon
        
#     def build(self, input_shape):
#         dim = input_shape[-1]
#         # Using the full, correct keyword arguments: 'name', 'shape', 'initializer'
#         self.gamma = self.add_weight(name='gamma', shape=(1,1,1,dim), initializer='ones')
#         self.beta = self.add_weight(name='beta', shape=(1,1,1,dim), initializer='zeros')

#     def call(self, inputs):
#         input_shape = tf.shape(inputs)
#         N, H, W, C = input_shape[0], input_shape[1], input_shape[2], input_shape[3]
#         group_size = C // self.groups
#         reshaped = tf.reshape(inputs, [N, H, W, self.groups, group_size])
#         mean, var = tf.nn.moments(reshaped, [1, 2, 4], keepdims=True)
#         normalized = (reshaped - mean) / tf.sqrt(var + self.epsilon)
#         return tf.reshape(normalized, input_shape) * self.gamma + self.beta


# # --- Step 3: Main Training Logic ---
# TASK_NAME = f"{CLASSES_TO_TRAIN[0]}_vs_{CLASSES_TO_TRAIN[1]}"
# print(f"\n{'='*20} STARTING TRAINING FOR: {TASK_NAME} {'='*20}")

# df = pd.read_excel(LABEL_PATH)
# df_task = df[df["Diagnosis"].isin(CLASSES_TO_TRAIN)].copy()
# if df_task['Diagnosis'].nunique() < 2: raise ValueError(f"Not enough classes found for task {TASK_NAME}")

# df_task['label'] = df_task['Diagnosis'].apply(lambda x: 0 if x == CLASSES_TO_TRAIN[0] else 1)
# label_dict = dict(zip(df_task["Patient ID"], df_task["label"]))

# # Undersampling
# df_class0 = df_task[df_task['label'] == 0]
# df_class1 = df_task[df_task['label'] == 1]
# min_size = min(len(df_class0), len(df_class1))
# df_balanced = pd.concat([df_class0.sample(n=min_size, random_state=42), df_class1.sample(n=min_size, random_state=42)])
# patient_ids_balanced, patient_labels_balanced = list(df_balanced["Patient ID"]), list(df_balanced["label"])

# train_pids, test_pids, _, _ = train_test_split(patient_ids_balanced, patient_labels_balanced, test_size=0.25, random_state=42, stratify=patient_labels_balanced)

# # Data Generation
# X_train, y_train, X_test, y_test = [], [], [], []
# for pid in patient_ids_balanced:
#     label = label_dict[pid]
#     paths = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) if f.startswith(str(pid)) and f.endswith('.wav')]
#     for path in paths:
#         try:
#             if pid in train_pids:
#                 X_train.append(extract_log_mel_spectrogram(path))
#                 y_train.append(label)
#                 for _ in range(3):
#                     X_train.append(extract_log_mel_spectrogram(path, do_augment=True))
#                     y_train.append(label)
#             elif pid in test_pids:
#                 X_test.append(extract_log_mel_spectrogram(path))
#                 y_test.append(label)
#         except Exception as e: print(f"Warning on {path}: {e}")

# # Normalization
# X_train_raw = np.array(X_train)
# mean_val, std_val = np.mean(X_train_raw, axis=0), np.std(X_train_raw, axis=0)
# model_name = MODEL_TO_SAVE_AS.replace('.keras', '')
# mean_path = os.path.join(SAVE_DIR, f"logmel_{model_name}_mean.npy")
# std_path = os.path.join(SAVE_DIR, f"logmel_{model_name}_std.npy")
# print(f"\nSaving normalization stats to {mean_path}...")
# np.save(mean_path, mean_val); np.save(std_path, std_val)

# X_train = (X_train_raw - mean_val) / (std_val + 1e-6)
# X_test = (np.array(X_test) - mean_val) / (std_val + 1e-6)
# X_train, X_test = X_train[..., np.newaxis], X_test[..., np.newaxis]
# y_train, y_test = np.array(y_train), np.array(y_test)

# print(f"Final Shapes - X_train: {X_train.shape}, X_test: {X_test.shape}")

# # Model Architecture
# model = Sequential([
#     Input(shape=(N_MELS, MAX_LEN, 1)),
#     Conv2D(16, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
#     GroupNormalization(), MaxPooling2D(), Dropout(0.3),
#     Conv2D(32, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
#     GroupNormalization(), MaxPooling2D(), Dropout(0.3),
#     Flatten(), Dense(64, activation='gelu', kernel_regularizer=l2(0.001)),
#     Dropout(0.5), Dense(1, activation='sigmoid')
# ])
# model.compile(optimizer=tf.keras.optimizers.Adam(INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
# model.summary()

# # Training
# print("\nStarting training...")
# model_path = os.path.join(SAVE_DIR, MODEL_TO_SAVE_AS)
# callbacks = [EarlyStopping('val_loss', patience=10, restore_best_weights=True), ModelCheckpoint(model_path, save_best_only=True), ReduceLROnPlateau(patience=4)]
# model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks)

# print(f"\n{'='*20} FINISHED TRAINING: {MODEL_TO_SAVE_AS} {'='*20}")


==================== STARTING TRAINING FOR: COPD2_vs_COPD3 ====================

Saving normalization stats to /home/punith/Desktop/cHEAL 2.o/All models/logmel_testing_2_3_mean.npy...
Final Shapes - X_train: (480, 20, 150, 1), X_test: (48, 20, 150, 1)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 20, 150, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_2           │ (None, 20, 150, 16)    │            32 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 10, 75, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_3           │ (None, 10, 75, 32)     │            64 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 5920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       378,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 383,905 (1.46 MB)

 Trainable params: 383,905 (1.46 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.5173 - loss: 1.4199 - val_accuracy: 0.5833 - val_loss: 0.9075 - learning_rate: 1.0000e-04
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.6890 - loss: 0.8796 - val_accuracy: 0.5833 - val_loss: 0.8804 - learning_rate: 1.0000e-04
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.7284 - loss: 0.7856 - val_accuracy: 0.4583 - val_loss: 0.9313 - learning_rate: 1.0000e-04
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - accuracy: 0.7607 - loss: 0.6787 - val_accuracy: 0.5417 - val_loss: 0.9099 - learning_rate: 1.0000e-04
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.8148 - loss: 0.5898 - val_accuracy: 0.5417 - val_loss: 0.9453 - learning_rate: 1.0000e-04
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.7812 - loss: 0.6287 - val_accuracy: 0.5000 - val_loss: 0.9022 - learning_rate: 1.0000e-04
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/st

In [3]:
# =========================================================================
#
#       Inference Script: Predict COPD Progression (Stage 2 vs. Stage 3)
#       This script loads the "testing_2_3_mel20.keras" model.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The audio file you want to predict ---
# !!! UPDATE THIS PATH TO YOUR AUDIO FILE !!!
AUDIO_FILE_TO_PREDICT = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/142_1b1_Pl_mc_LittC2SE.wav"

# --- Parameters that MUST match the training script ---
N_MELS = 20  # CRUCIAL: Must be 20 to match the model's input shape
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'} # Model was trained with COPD2=0, COPD3=1

def predict_progression_stage(audio_path):
    """
    Loads the trained 2-vs-3 (N_MELS=20) model and predicts the stage.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        print("Ensure 'testing_2_3_mel20.keras' and its .npy files are present.")
        return

    # --- 2. Process the New Audio File ---
    print(f"\n--- Processing audio file: {os.path.basename(audio_path)} ---")
    try:
        y, sr = librosa.load(audio_path, sr=None)
        
        # This will now create a (20, t) spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS) 
        log_mel = librosa.power_to_db(mel_spec)
        
        if log_mel.shape[1] < MAX_LEN:
            log_mel = np.pad(log_mel, ((0, 0), (0, MAX_LEN - log_mel.shape[1])), mode='constant')
        else:
            log_mel = log_mel[:, :MAX_LEN]
            
        # The normalization will now correctly be (20, 150) - (20, 150)
        log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
        log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
        
    except Exception as e:
        print(f"❌ ERROR: Failed to process the audio file. Error: {e}")
        return

    # --- 3. Make the Prediction ---
    print("\n--- Making prediction... ---")
    try:
        prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
        if prediction_prob >= 0.5:
            predicted_class_name = LABELS[1] # 'COPD3'
            confidence = prediction_prob
        else:
            predicted_class_name = LABELS[0] # 'COPD2'
            confidence = 1 - prediction_prob
    except Exception as e:
        print(f"❌ ERROR: Model failed to predict. Error: {e}")
        return

    # --- 4. Display the Result ---
    print("\n" + "="*50)
    print("--- 🩺 Progression Prediction Result (Stage 2 vs. 3) ---")
    print(f"The model predicts the stage for this recording is:")
    print(f"    >> {predicted_class_name} <<")
    print(f"    Confidence: {confidence * 100:.2f}%")
    print("="*50)
    print(f"Raw Model Output (0.0 ≈ COPD2, 1.0 ≈ COPD3): {prediction_prob:.4f}")

# --- Run the prediction function ---
if __name__ == "__main__":
    if os.path.exists(AUDIO_FILE_TO_PREDICT):
        predict_progression_stage(AUDIO_FILE_TO_PREDICT)
    else:
        print(f"❌ ERROR: The audio file does not exist at '{AUDIO_FILE_TO_PREDICT}'")

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Processing audio file: 142_1b1_Pl_mc_LittC2SE.wav ---

--- Making prediction... ---

--- 🩺 Progression Prediction Result (Stage 2 vs. 3) ---
The model predicts the stage for this recording is:
    >> COPD3 <<
    Confidence: 77.04%
Raw Model Output (0.0 ≈ COPD2, 1.0 ≈ COPD3): 0.7704


114

In [4]:
# =========================================================================
#
#       Patient Analysis Script for "testing_2_3_mel20.keras"
#
# This script loads the trained 2-vs-3 model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 2 to Stage 3.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files from your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "114"

# --- Parameters that MUST match the training script ---
N_MELS = 20
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (2 vs 3) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 114 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 5 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 114_1b4_Al_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 54.24%
  -> File: 114_1b4_Ar_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 64.47%
  -> File: 114_1b4_Lr_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 62.04%
  -> File: 114_1b4_Pl_mc_AKGC417L.wav               | Predicted: COPD3    | Confidence: 51.89%
  -> File: 114_1b4_Pr_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 73.99%

--- 📋 Final Analysis Summary for Patient 114 (2 vs 3) ---
Based on a majority vote of all 5 recordings, the most likely stage is:
    >> COPD2 <<
    (This received 4 out of 5 votes,

142

In [5]:
# =========================================================================
#
#       Patient Analysis Script for "testing_2_3_mel20.keras"
#
# This script loads the trained 2-vs-3 model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 2 to Stage 3.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files from your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "142"

# --- Parameters that MUST match the training script ---
N_MELS = 20
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (2 vs 3) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 142 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 1 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 142_1b1_Pl_mc_LittC2SE.wav               | Predicted: COPD3    | Confidence: 77.04%

--- 📋 Final Analysis Summary for Patient 142 (2 vs 3) ---
Based on a majority vote of all 1 recordings, the most likely stage is:
    >> COPD3 <<
    (This received 1 out of 1 votes, for a 100.00% consensus.)


110

In [6]:
# =========================================================================
#
#       Patient Analysis Script for "testing_2_3_mel20.keras"
#
# This script loads the trained 2-vs-3 model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 2 to Stage 3.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files from your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "110"

# --- Parameters that MUST match the training script ---
N_MELS = 20
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (2 vs 3) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 110 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 5 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 110_1b1_Pr_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 86.97%
  -> File: 110_1p1_Al_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 81.73%
  -> File: 110_1p1_Ll_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 98.96%
  -> File: 110_1p1_Lr_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 91.28%
  -> File: 110_1p1_Pr_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 59.50%

--- 📋 Final Analysis Summary for Patient 110 (2 vs 3) ---
Based on a majority vote of all 5 recordings, the most likely stage is:
    >> COPD3 <<
    (This received 5 out of 5 votes,

106

In [7]:
# =========================================================================
#
#       Patient Analysis Script for "testing_2_3_mel20.keras"
#
# This script loads the trained 2-vs-3 model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 2 to Stage 3.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files from your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "106"

# --- Parameters that MUST match the training script ---
N_MELS = 20
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (2 vs 3) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 106 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 2 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 106_2b1_Pl_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 63.67%
  -> File: 106_2b1_Pr_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 68.32%

--- 📋 Final Analysis Summary for Patient 106 (2 vs 3) ---
Based on a majority vote of all 2 recordings, the most likely stage is:
    >> COPD2 <<
    (This received 2 out of 2 votes, for a 100.00% consensus.)


221

In [8]:
# =========================================================================
#
#       Patient Analysis Script for "testing_2_3_mel20.keras"
#
# This script loads the trained 2-vs-3 model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 2 to Stage 3.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files from your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "221"

# --- Parameters that MUST match the training script ---
N_MELS = 20
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (2 vs 3) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 221 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 12 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 221_2b1_Al_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 70.21%
  -> File: 221_2b1_Ar_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 73.59%
  -> File: 221_2b1_Lr_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 63.57%
  -> File: 221_2b1_Pl_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 75.63%
  -> File: 221_2b2_Al_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 65.51%
  -> File: 221_2b2_Ar_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 69.18%
  -> File: 221_2b2_Lr_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence:

222

In [9]:
# =========================================================================
#
#       Patient Analysis Script for "testing_2_3_mel20.keras"
#
# This script loads the trained 2-vs-3 model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 2 to Stage 3.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files from your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "222"

# --- Parameters that MUST match the training script ---
N_MELS = 20
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (2 vs 3) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 222 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 3 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 222_1b1_Ar_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 92.91%
  -> File: 222_1b1_Lr_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 53.74%
  -> File: 222_1b1_Pr_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 69.98%

--- 📋 Final Analysis Summary for Patient 222 (2 vs 3) ---
Based on a majority vote of all 3 recordings, the most likely stage is:
    >> COPD3 <<
    (This received 3 out of 3 votes, for a 100.00% consensus.)


223

In [10]:
# =========================================================================
#
#       Patient Analysis Script for "testing_2_3_mel20.keras"
#
# This script loads the trained 2-vs-3 model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 2 to Stage 3.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files from your "2 vs 3, N_MELS=20" training run ---
MODEL_PATH = "testing_2_3_mel20.keras"
MEAN_PATH = "logmel_testing_2_3_mel20_mean.npy"
STD_PATH = "logmel_testing_2_3_mel20_std.npy"

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "223"

# --- Parameters that MUST match the training script ---
N_MELS = 20
MAX_LEN = 150
LABELS = {0: 'COPD2', 1: 'COPD3'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 2-vs-3 (N_MELS=20) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (2 vs 3) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 2-vs-3 (N_MELS=20) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 223 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 6 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 223_1b1_Al_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 57.13%
  -> File: 223_1b1_Ar_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 65.16%
  -> File: 223_1b1_Ll_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 71.53%
  -> File: 223_1b1_Lr_sc_Meditron.wav               | Predicted: COPD3    | Confidence: 53.84%
  -> File: 223_1b1_Pl_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 50.99%
  -> File: 223_1b1_Pr_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 66.13%

--- 📋 Final Analysis Summary for Patient 223 (2 vs 3) ---
Based on a majority vote of a